# Quantum gate optimization

In [1]:
import numpy as np

from qoptcraft.algebra import photon_basis
from qoptcraft.math import haar_random_unitary
from qoptcraft.optimization import HeraldedGatePrep, PiecewiseInfidelityCost, BFGS

import sys, os
sys.path.insert(0, os.path.abspath('..'))
from results_table import benchmarks_table

## CZ Gate

In [2]:
modes = 6
photons = 4
# Computational basis: dual-rail encoding |00⟩,|01⟩,|10⟩,|11⟩ + ancilla (1,1)
input_basis = [(1, 0, 1, 0, 1, 1), (1, 0, 0, 1, 1, 1), (0, 1, 1, 0, 1, 1), (0, 1, 0, 1, 1, 1)]
# Subspace with heralding pattern (1,1)
output_basis = [state for state in photon_basis(modes, photons) if state[4] == 1 and state[5] == 1]

# Rectangular target: CZ diagonal [1,1,1,-1] on dual-rail rows, 0 on leakage rows
target_gate = np.zeros((len(output_basis), len(input_basis)), dtype=complex)
for col, state in enumerate(input_basis):
    target_gate[output_basis.index(state), col] = [1., 1., 1., -1.][col]

In [3]:
problem = HeraldedGatePrep(target_gate, input_basis, output_basis)
S0 = haar_random_unitary(modes)
optimizer = BFGS(max_iter=1000, tol_grad=1e-10)
cost = PiecewiseInfidelityCost(problem)

In [ ]:
N_RUNS = 10000
results = [optimizer.minimize(cost, cost.manifold.random_point(), verbose=False) for _ in range(N_RUNS)]
all_results = {"CZ gate": results}

In [5]:
PROB_THRESH = 0.073
benchmarks_table(all_results, problem, PROB_THRESH)

,Runs,Converged (%),Mean time (s),Mean iters
Optimizer,,,,
CZ gate,10000,6.8%,0.004,45


## CCZ Gate

In [6]:
modes = 12
photons = 6
herald = (1, 1, 1, 0, 0, 0)
# Computational basis: dual-rail encoding |000⟩...|111⟩ + ancilla (1,1,1,0,0,0)
input_basis = [
    (1,0,1,0,1,0) + herald,  # |000⟩
    (1,0,1,0,0,1) + herald,  # |001⟩
    (1,0,0,1,1,0) + herald,  # |010⟩
    (1,0,0,1,0,1) + herald,  # |011⟩
    (0,1,1,0,1,0) + herald,  # |100⟩
    (0,1,1,0,0,1) + herald,  # |101⟩
    (0,1,0,1,1,0) + herald,  # |110⟩
    (0,1,0,1,0,1) + herald,  # |111⟩
]
# Subspace with heralding pattern (1,1,1,0,0,0): the whole ancilla block must match, empty
# modes included, since a photon reaching modes 9-11 is a detected failure, not a leakage row.
output_basis = [state for state in photon_basis(modes, photons) if state[6:] == herald]

# Rectangular target: CCZ diagonal [1,1,1,1,1,1,1,-1] on dual-rail rows, 0 on leakage rows
target_gate = np.zeros((len(output_basis), len(input_basis)), dtype=complex)
for col, state in enumerate(input_basis):
    target_gate[output_basis.index(state), col] = [1., 1., 1., 1., 1., 1., 1., -1.][col]

In [7]:
problem = HeraldedGatePrep(target_gate, input_basis, output_basis)
S0 = haar_random_unitary(modes)
optimizer = BFGS(max_iter=5000, tol_grad=1e-10)
cost = PiecewiseInfidelityCost(problem)

In [ ]:
N_RUNS = 10000
results = [optimizer.minimize(cost, cost.manifold.random_point(), verbose=False) for _ in range(N_RUNS)]
all_results = {"CCZ gate": results}

In [ ]:
PROB_THRESH = 0.0033  # the known Toffoli/CCZ optimum sits at P ≈ 0.34%
benchmarks_table(all_results, problem, PROB_THRESH)

,Runs,Converged (%),Mean time (s),Mean iters
Optimizer,,,,
CCZ gate,10000,0.6%,0.299,462
